# 04 · Edges, START, END, compile and invoke

**Goal:** by the end you can wire nodes together with edges, compile the builder into a graph, run it with
`invoke`, and print a diagram of it.

## The idea in plain English

An **edge** is an arrow from one node to the next. You add one with `add_edge("from", "to")`. Two special names
mark the ends of the graph: **`START`** (where the input comes in) and **`END`** (where the final state comes out).

When the wiring is done you call **`compile()`**. Compiling checks the wiring (does every edge point at a real
node? is there a way in from `START`?) and gives you a **compiled graph** you can run. Running it is called
**`invoke`**: you pass in a starting state, LangGraph runs the nodes in order, merges each update, and hands back the
final state.

**Analogy:** planning a train line. Adding edges is drawing tracks on the map. Compiling is the inspector checking
that every track ends at a real station. Invoking is running a train from the first station to the last.

## Picture

First the smallest possible graph, then a three-step pipeline.

```mermaid
flowchart LR
    S1([START]) --> add --> E1([END])
```

```mermaid
flowchart LR
    S2([START]) --> parse --> calculate --> format --> E2([END])
```

## Part 1: The smallest graph, START → add → END

### Step 1: Imports, and what START and END really are

`START` and `END` are just special strings. LangGraph reserves those two names for the entry and the exit.

In [1]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END

print("START is", repr(START))
print("END is  ", repr(END))

START is '__start__'
END is   '__end__'


### Step 2: State and node, as in lesson 03

In [2]:
class CalcState(TypedDict):
    a: float
    b: float
    result: float


def add_node_fn(state: CalcState) -> dict:
    return {"result": state["a"] + state["b"]}

### Step 3: Register the node and add two edges

`add_edge(START, "add")` means "the input goes to `add` first". `add_edge("add", END)` means "after `add`, stop".
The builder keeps its edges as `(from, to)` pairs, which we print.

In [3]:
builder = StateGraph(CalcState)
builder.add_node("add", add_node_fn)

builder.add_edge(START, "add")
builder.add_edge("add", END)

print("nodes:", list(builder.nodes))
print("edges:", sorted(builder.edges))

nodes: ['add']
edges: [('__start__', 'add'), ('add', '__end__')]


### Step 4: Compile

`compile()` checks the wiring and returns a new object, the compiled graph. The builder was for drawing. The
compiled graph is for running.

In [4]:
graph = builder.compile()
print(type(graph).__name__)

CompiledStateGraph


### Step 5: Draw the graph

`graph.get_graph().draw_mermaid()` returns the diagram as **Mermaid** text (a small language for drawing flowcharts).
It works offline. In the output, `__start__` and `__end__` are `START` and `END`. The lines starting with
`classDef` and the `config` block only set colours and line style, so you can skip them when reading.

In [5]:
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	add(add)
	__end__([<p>__end__</p>]):::last
	__start__ --> add;
	add --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



If your Jupyter can display Mermaid (JupyterLab 4.1 or newer can), wrapping the same text in a ```` ```mermaid ````
block draws it as a picture. If you only see text, that's fine: the printed version above is the same diagram.

In [6]:
from IPython.display import Markdown, display

display(Markdown(f"```mermaid\n{graph.get_graph().draw_mermaid()}\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	add(add)
	__end__([<p>__end__</p>]):::last
	__start__ --> add;
	add --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

### Step 6: Invoke

`invoke` takes the starting state and returns the final state. We don't pass `result`: the `add` node fills it in.

In [7]:
final = graph.invoke({"a": 3.0, "b": 4.0})
print(final)

{'a': 3.0, 'b': 4.0, 'result': 7.0}


A compiled graph can be invoked again and again. Each call starts from the input you give it.

In [8]:
for a, b in [(1.0, 2.0), (10.0, -4.0), (0.5, 0.25)]:
    print(graph.invoke({"a": a, "b": b}))

{'a': 1.0, 'b': 2.0, 'result': 3.0}
{'a': 10.0, 'b': -4.0, 'result': 6.0}
{'a': 0.5, 'b': 0.25, 'result': 0.75}


## Part 2: A three-step pipeline, parse → calculate → format

Now the calculator from lesson 01, but run by LangGraph. The user types text such as `"3 + 4"`, and three nodes turn
it into `"3 + 4 = 7"`.

### Step 7: A bigger state

The state now holds the text the user typed, the pieces `parse` finds, the `result`, and the final `answer`.

In [9]:
class PipelineState(TypedDict):
    expression: str  # what the user typed, e.g. "3 + 4"
    a: float
    op: str  # "+", "-", "*" or "/"
    b: float
    result: float
    answer: str  # e.g. "3 + 4 = 7"

### Step 8: The three nodes

Each one returns only the keys it adds. `calculate` still chooses the operation with `if/elif`. Lesson 05 turns
those choices into separate nodes so you can see them in the diagram.

In [10]:
def parse(state: PipelineState) -> dict:
    left, op, right = state["expression"].split()
    return {"a": float(left), "op": op, "b": float(right)}


def calculate(state: PipelineState) -> dict:
    a, op, b = state["a"], state["op"], state["b"]
    if op == "+":
        return {"result": a + b}
    elif op == "-":
        return {"result": a - b}
    elif op == "*":
        return {"result": a * b}
    elif op == "/":
        return {"result": a / b}
    raise ValueError(f"Unknown operator: {op}")

`format_answer` turns the result into the sentence the user sees, as in lesson 01.

In [11]:
def format_answer(state: PipelineState) -> dict:
    return {"answer": f"{state['expression']} = {state['result']:g}"}

### Step 9: Wire, compile and draw

Four edges make a straight line: `START → parse → calculate → format → END`. The node is named `"format"`, and the
function behind it is `format_answer`. A node's name and its function's name don't have to match.

In [12]:
builder = StateGraph(PipelineState)
builder.add_node("parse", parse)
builder.add_node("calculate", calculate)
builder.add_node("format", format_answer)

builder.add_edge(START, "parse")
builder.add_edge("parse", "calculate")
builder.add_edge("calculate", "format")
builder.add_edge("format", END)

pipeline = builder.compile()
print(pipeline.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parse(parse)
	calculate(calculate)
	format(format)
	__end__([<p>__end__</p>]):::last
	__start__ --> parse;
	calculate --> format;
	parse --> calculate;
	format --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### Step 10: Invoke the pipeline

We pass only `expression`. Every other key is filled in by a node on the way through. The full final state comes
back, so we print it, then print just the `answer`.

In [13]:
final = pipeline.invoke({"expression": "3 + 4"})
print(final)
print(final["answer"])

{'expression': '3 + 4', 'a': 3.0, 'op': '+', 'b': 4.0, 'result': 7.0, 'answer': '3 + 4 = 7'}
3 + 4 = 7


The same compiled pipeline handles other expressions too. This time we print only the `answer`.

In [14]:
for expression in ["10 - 2.5", "6 * 7", "10 / 4"]:
    print(pipeline.invoke({"expression": expression})["answer"])

10 - 2.5 = 7.5
6 * 7 = 42
10 / 4 = 2.5


## Part 3: Compiling checks the wiring

Mistakes in the wiring are caught by `compile()`, before anything runs. Below, an edge points at `"calc"`, a node
that doesn't exist (the real one is `"calculate"`). We catch the error so the notebook keeps running, and print it.

In [15]:
broken = StateGraph(PipelineState)
broken.add_node("parse", parse)
broken.add_node("calculate", calculate)
broken.add_edge(START, "parse")
broken.add_edge("parse", "calc")  # typo: there is no node called "calc"

try:
    broken.compile()
except ValueError as err:
    print("compile() refused:", err)

compile() refused: Found edge ending at unknown node `calc`


A graph also needs a way in. Without any edge from `START`, LangGraph doesn't know which node to run first.

In [16]:
no_entry = StateGraph(PipelineState)
no_entry.add_node("parse", parse)
no_entry.add_edge("parse", END)  # but nothing goes from START to parse

try:
    no_entry.compile()
except ValueError as err:
    print("compile() refused:", err)

compile() refused: Graph must have an entrypoint: add at least one edge from START to another node


## What just happened

- **Step 1** showed that `START` and `END` are the reserved strings `'__start__'` and `'__end__'`.
- **Steps 3–4** added the edges `START → add → END` and compiled the builder into a `CompiledStateGraph`.
- **Step 5** printed the diagram: `__start__ --> add` and `add --> __end__`.
- **Step 6** invoked the graph with `{'a': 3.0, 'b': 4.0}` and got back `{'a': 3.0, 'b': 4.0, 'result': 7.0}`. The
  node returned only `result`; LangGraph merged it in.
- **Steps 7–10** built the `parse → calculate → format` pipeline. From just `{'expression': '3 + 4'}` it returned
  the full state, ending with `'answer': '3 + 4 = 7'`.
- **Part 3** showed `compile()` rejecting an edge to the unknown node `calc`, and a graph with no entry from `START`.

## Common mistakes

1. **Invoking the builder instead of the compiled graph.** The builder can't run:
   `AttributeError: 'StateGraph' object has no attribute 'invoke'`.
   Fix: `graph = builder.compile()`, then `graph.invoke(...)`.
2. **A typo in a key a node returns.** If `calculate` returns `{"reslt": 7.0}`, there's **no error** at that point.
   LangGraph ignores keys that aren't in the state schema. The next node then fails when it reads the real key:
   `KeyError: 'result'`.
   Fix: check that returned keys match the names in your state class exactly.
3. **Leaving out an input the first node needs.** `graph.invoke({"a": 3.0})` runs `add`, which reads `state["b"]`:
   `KeyError: 'b'`.
   Fix: pass every key the first node reads.

## Try it

1. Add a node `square` to the Part 1 graph so it runs `START → add → square → END`. It should replace `result` with
   `result * result`. What does `invoke({"a": 3.0, "b": 4.0})` return? Print the diagram too.
2. In the pipeline, the edge `parse → calculate` is replaced by an edge `parse → END`. Which nodes run now,
   and what does `invoke({"expression": "3 + 4"})` return? Predict first, then check.

Answers are in [`solutions/exercises_solutions.ipynb`](../solutions/exercises_solutions.ipynb).

## Key terms

New terms from this lesson, also in [GLOSSARY.md](../GLOSSARY.md):

- **`add_edge`**: adds a fixed arrow from one node to another.
- **`START` / `END`**: reserved node names for the graph's entry and exit (`'__start__'` and `'__end__'`).
- **`compile()`**: checks the builder's wiring and returns a runnable compiled graph.
- **Compiled graph**: the runnable object `compile()` returns. It has `invoke`, `get_graph` and more.
- **`invoke`**: runs the compiled graph from a starting state and returns the final state.
- **Mermaid / `draw_mermaid()`**: a text format for diagrams, and the method that prints a graph in it.